# GammaGateR Per-Marker Thresholds

Fits a zero-inflated two-component gamma mixture to each protein marker with [GammaGateR](https://github.com/JiangmeiRubyXiong/GammaGateR) and turns the posterior probability into a raw-intensity threshold.

The cut is the largest raw intensity whose posterior is still below 0.5, so the existing rule `X > threshold` matches the paper's `posterior >= 0.5` call. Results go to `output/gating_thresholds_gammagater.csv` and are plotted against the current `output/gating_thresholds.csv`. This notebook does not overwrite the production gates.

GammaGateR still gates each marker on its own. A cell that is bright on two incompatible markers stays double-positive. Use this to place each marker's cut, not to clear `Artifact_Spillover`.

**Kernel:** `napari-env`. GammaGateR itself runs in a separate conda env named `gammagater`.

---

## Install

Create a new env. Do not install this into `napari-env`. GammaGateR is an R package (and it depends on `cfGMM`, also R, also GitHub-only). Mixing that stack into the imaging env means an R upgrade can break napari, cellpose, and anndata.

System R (`/usr/bin/R`, 4.6) is also the wrong place to install it. The package and its imports were written for R 4.3–4.4, and a user-library install there is harder to throw away if a dependency fails to compile. The env below pins R 4.4 from conda-forge and preinstalls the CRAN imports so the GitHub step only builds `cfGMM` and `GammaGateR`.

```bash
conda env create -f envs/gammagater.yml
export R_PROFILE_USER=/dev/null R_ENVIRON_USER=/dev/null
RS=~/miniforge3/envs/gammagater/bin/Rscript
LIB=~/miniforge3/envs/gammagater/lib/R/library
$RS -e 'install.packages(c("shapefiles", "fossil"), repos = "https://cloud.r-project.org", lib = "'$LIB'")'
$RS -e 'remotes::install_github("JiangmeiRubyXiong/cfGMM", upgrade = "never", dependencies = FALSE, lib = "'$LIB'")'
$RS -e 'remotes::install_github("JiangmeiRubyXiong/GammaGateR", upgrade = "never", dependencies = FALSE, build_vignettes = FALSE, lib = "'$LIB'")'
```

`~/.Rprofile` prepends `~/R/aarch64-unknown-linux-gnu-library/4.6` to every R session, and `~/.Renviron` points `R_LIBS_USER` at the `omicslogic-sc` env. Packages in those libraries were built for R 4.6. This env is R 4.4, so loading them fails (`data.table` reports `undefined symbol: R_duplicateAsResizable`). The two `export`s above are required while installing: `install.packages` and `remotes` otherwise search the R 4.6 library and hit the same error. `scripts/gammagater_fit.R` then calls `.libPaths(file.path(R.home(), "library"))` before `library()`, so the fit stays on this env even when a later `Rscript` still reads `~/.Rprofile`. The notebook's `r_env()` sets the same two variables for the version check.

`r-fossil` is absent from conda-forge on linux-aarch64 (it is built for x86_64 only). `fossil` itself is pure R; its compiled dependencies (`sp`, `maps`, `foreign`) are in the env file, and the `install.packages` line above builds `shapefiles` and `fossil` from CRAN.

`dependencies = FALSE` is intentional: the other imports are already in the env file. If `library(GammaGateR)` reports a missing package, install that one package into this env rather than flipping the flag and letting remotes rebuild ggplot2.

The first code cell checks that `~/miniforge3/envs/gammagater/bin/Rscript` can load the package.


## 1. Imports and paths

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import anndata as ad

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONTINUOUS_H5AD = PROJECT_ROOT / "output" / "celldive_protein_matrix_continuous.h5ad"
PRODUCTION_CSV = PROJECT_ROOT / "output" / "gating_thresholds.csv"
INPUT_CSV = PROJECT_ROOT / "output" / "gammagater_input.csv"
BOUNDARY_JSON = PROJECT_ROOT / "output" / "gammagater_boundaries.json"
SUMMARY_CSV = PROJECT_ROOT / "output" / "gating_thresholds_gammagater.csv"
RSCRIPT = Path.home() / "miniforge3" / "envs" / "gammagater" / "bin" / "Rscript"
FIT_SCRIPT = PROJECT_ROOT / "scripts" / "gammagater_fit.R"
SLIDE_ID = "SLIDE-045"
PROB_CUT = 0.5

def r_env():
    env = os.environ.copy()
    env["PATH"] = str(RSCRIPT.parent) + os.pathsep + env.get("PATH", "")
    # Skip ~/.Rprofile and ~/.Renviron so R 4.4 does not load the R 4.6 user library.
    env["R_PROFILE_USER"] = "/dev/null"
    env["R_ENVIRON_USER"] = "/dev/null"
    return env

print(f"Project root : {PROJECT_ROOT}")
print(f"Rscript      : {RSCRIPT}  ({'found' if RSCRIPT.exists() else 'MISSING'})")
print(f"Summary out  : {SUMMARY_CSV}")

In [ ]:
if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"{RSCRIPT} not found.\n"
        "Create the env first:\n"
        "  conda env create -f envs/gammagater.yml\n"
        "then install cfGMM and GammaGateR (commands are in the notebook header)."
    )

check = subprocess.run(
    [str(RSCRIPT), "-e", 'cat(as.character(packageVersion("GammaGateR")), "\n")'],
    capture_output=True,
    text=True,
    env=r_env(),
)
if check.returncode != 0:
    raise RuntimeError(
        "GammaGateR is not installed in the gammagater env.\n"
        + check.stderr
    )
print("GammaGateR", check.stdout.strip())

## 2. Load continuous intensities

Same input as `determine_binary_threshold.ipynb`. DAPI and DAPI2 are skipped: cells are already the Cellpose nuclei. Set `SUBSAMPLE` to an integer for a smoke test; leave it `None` for the threshold you would actually compare to production.

In [ ]:
SKIP_MARKERS = {"DAPI", "DAPI2"}
SUBSAMPLE = None  # e.g. 50_000 for a first-pass timing check

if not CONTINUOUS_H5AD.exists():
    raise FileNotFoundError(
        f"{CONTINUOUS_H5AD} not found. Build it from determine_binary_threshold.ipynb first."
    )

adata = ad.read_h5ad(CONTINUOUS_H5AD)
X = np.asarray(adata.X, dtype=np.float64)
var_names = list(adata.var_names)
if len(np.unique(X[:1000])) <= 2:
    raise ValueError(f"{CONTINUOUS_H5AD.name} looks binary. Use the continuous matrix.")

gating_markers = [n for n in var_names if n not in SKIP_MARKERS]
col = [var_names.index(m) for m in gating_markers]
Xg = X[:, col]
if SUBSAMPLE is not None and SUBSAMPLE < Xg.shape[0]:
    rng = np.random.default_rng(0)
    take = rng.choice(Xg.shape[0], SUBSAMPLE, replace=False)
    take.sort()
    Xg = Xg[take]
    print(f"Subsampled to {SUBSAMPLE:,} cells. Thresholds from this run are a smoke test.")

print(f"Cells: {Xg.shape[0]:,}   Markers: {len(gating_markers)}")
print(gating_markers)

## 3. Quantile boundaries

Optional. These are constraints on where each component's **mode** may sit, as quantiles of that marker's normalized intensity. They are not the positivity threshold.

Leave the dict empty for the first fit. After looking at the histograms, pin a marker whose positive component collapsed onto the background. Example for a rare positive population: the negative mode stays left of the 80th percentile and the positive mode stays right of the 90th.

```python
QUANTILE_BOUNDARIES = {
    "CD20": [[0.0, 0.80], [0.90, 1.0]],
}
```

Abundant markers (CD45, VIM, PANCK) often have a positive mode well below the 90th percentile. A rare-cell constraint on those markers will not converge.

In [ ]:
QUANTILE_BOUNDARIES = {}

unknown = set(QUANTILE_BOUNDARIES) - set(gating_markers)
if unknown:
    raise KeyError(f"Boundary markers not in the matrix: {sorted(unknown)}")

BOUNDARY_JSON.write_text(json.dumps(QUANTILE_BOUNDARIES))
print(f"Wrote {BOUNDARY_JSON.name}: {QUANTILE_BOUNDARIES or '(unconstrained)'}")

## 4. Fit

The R script normalizes each marker the way the GammaGateR vignette does: `log10(1 + x / mean(x[x > 0]))`, fits one slide, and maps the 0.5 posterior crossing back onto the raw intensity. Full-slide runtime is several minutes.

In [ ]:
input_df = pd.DataFrame(Xg, columns=gating_markers)
input_df.insert(0, "cell_id", np.arange(len(input_df)))
input_df.to_csv(INPUT_CSV, index=False)
print(f"Wrote {INPUT_CSV}  ({INPUT_CSV.stat().st_size / 1e6:.0f} MB)")

cmd = [
    str(RSCRIPT), str(FIT_SCRIPT),
    "--input", str(INPUT_CSV),
    "--summary", str(SUMMARY_CSV),
    "--boundaries", str(BOUNDARY_JSON),
    "--slide", SLIDE_ID,
    "--prob", str(PROB_CUT),
]
proc = subprocess.run(cmd, capture_output=True, text=True, env=r_env())
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr)
if proc.stderr.strip():
    print(proc.stderr)

## 5. Compare with the production gates

`positive_rate` is the fraction of fitted cells with posterior ≥ 0.5. `production_positive_rate` applies the current CSV with the same `X > threshold` rule on those same cells. A marker that did not converge has no GammaGateR threshold; leave the production gate in place for that marker.

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
production = pd.read_csv(PRODUCTION_CSV).set_index("marker")["threshold"]

rows = []
for rec in summary.itertuples(index=False):
    j = gating_markers.index(rec.marker)
    v = Xg[:, j]
    prod_thr = float(production.get(rec.marker, np.nan))
    rows.append({
        "marker": rec.marker,
        "converged": bool(rec.converged),
        "gammagater_threshold": rec.threshold,
        "gammagater_positive_rate": float(np.mean(v > rec.threshold)) if np.isfinite(rec.threshold) else np.nan,
        "production_threshold": prod_thr,
        "production_positive_rate": float(np.mean(v > prod_thr)) if np.isfinite(prod_thr) else np.nan,
        "lambda_pos": rec.lambda_pos,
        "mode_neg": rec.mode_neg,
        "mode_pos": rec.mode_pos,
        "message": rec.message,
    })

compare = pd.DataFrame(rows)
compare["rate_delta"] = compare["gammagater_positive_rate"] - compare["production_positive_rate"]
with pd.option_context("display.max_rows", 30, "display.width", 140, "display.float_format", lambda v: f"{v:.4g}"):
    display_cols = [
        "marker", "converged",
        "gammagater_threshold", "production_threshold",
        "gammagater_positive_rate", "production_positive_rate", "rate_delta",
        "mode_neg", "mode_pos", "message",
    ]
    print(compare[display_cols].to_string(index=False))

n_fail = int((~compare["converged"]).sum())
print(f"\n{n_fail} marker(s) did not converge.")

In [ ]:
n = len(gating_markers)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.6 * nrows), squeeze=False)

for i, marker in enumerate(gating_markers):
    ax = axes[i // ncols, i % ncols]
    v = Xg[:, i]
    v = v[np.isfinite(v) & (v > 0)]
    ax.hist(np.log1p(v), bins=80, color="0.75", density=True)
    rec = compare.loc[compare["marker"] == marker].iloc[0]
    if np.isfinite(rec["production_threshold"]) and rec["production_threshold"] > 0:
        ax.axvline(np.log1p(rec["production_threshold"]), color="#d95f02", lw=1.2, label="production")
    if np.isfinite(rec["gammagater_threshold"]) and rec["gammagater_threshold"] > 0:
        ax.axvline(np.log1p(rec["gammagater_threshold"]), color="#1b9e77", lw=1.2, label="GammaGateR")
    ax.set_title(marker, fontsize=10)
    if i == 0:
        ax.legend(fontsize=8, frameon=False)

for j in range(n, nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

fig.suptitle("log1p intensity, production gate (orange) vs GammaGateR 0.5 posterior (green)", y=1.01)
fig.tight_layout()
plt.show()

## 6. What to do with the table

Inspect markers where the two positive rates disagree, or where `mode_pos` sits on top of `mode_neg` (the unconstrained fit put both components on the background). Set a quantile boundary for those markers and re-run from section 3.

When a marker looks right, copy its threshold into `MANUAL_OVERRIDES` in `determine_binary_threshold.ipynb`, or replace that row in `output/gating_thresholds.csv` by hand. This notebook will not write the production CSV.

Failed or nearly unimodal stains (PDGFRA, EPCAM on this slide) will not become real signal because the mixture found a cleaner cut. If both modes overlap and the posterior stays near 0, keep the production gate and treat the channel as failed.
